# Feature Engineering

Notebook 01 found that no raw column has |r| > 0.05 with Yield. So a model that only sees the raw 46 columns will never reach a useful R-squared. In this notebook we build two kinds of derived features:

1. Classical agronomic ratios (NPK ratio, soil health index, weather stress, vegetation index).
2. Latent proxies built as weighted z-scores of related raw columns (climate, soil, management, water, terrain) plus three tanh-bounded interactions of those proxies.

We then check whether the new features actually correlate with Yield, do the train/test split, and save a fitted preprocessor.

## Imports

In [1]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
RAW_PATH = ROOT / 'data' / 'raw' / 'Agri_yield_prediction.csv'
PROC_DIR = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'models'
FIG_DIR = ROOT / 'figures'
for d in (PROC_DIR, MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## Load raw data

In [2]:
df_raw = pd.read_csv(RAW_PATH)
print('Raw shape:', df_raw.shape)
df_raw.head(3)

Raw shape: (36000, 46)


,Temperature,Humidity,Rainfall,Soil_Type,pH,EC,OC,N,P,K,...,Planting_Date,Harvest_Date,Growth_Stage,Irrigation_Frequency,Fertilizer_Type,Pesticide_Usage,Yield,Region,Season,Year
0,20.217609,71.563782,48.837768,Clayey,6.467371,1.317873,0.656934,138.565862,65.148562,197.716636,...,2023-10-19,2024-01-29,Vegetative,4,Organic,Low,5.340428,West,Rabi,2023
1,34.144330,62.473178,253.900950,Sandy,7.087405,1.564200,0.748017,82.633876,37.236441,99.902932,...,2009-08-23,2009-11-13,Reproductive,4,Mixed,High,6.633464,North,Kharif,2009
2,20.842568,60.215798,128.300095,Loamy,6.685763,1.162799,1.185390,140.045407,72.826335,151.476596,...,2022-12-23,2023-04-11,Vegetative,5,Mixed,High,7.088017,South,Rabi,2022


## Build engineered features

In [ ]:
def add_engineered_features(df):
    out = df.copy()
    out['Planting_Date'] = pd.to_datetime(out['Planting_Date'])
    out['Harvest_Date'] = pd.to_datetime(out['Harvest_Date'])
    out['Crop_Duration'] = (out['Harvest_Date'] - out['Planting_Date']).dt.days
    out['Planting_Month'] = out['Planting_Date'].dt.month
    out['Harvest_Month'] = out['Harvest_Date'].dt.month
    out = out.drop(columns=['Planting_Date', 'Harvest_Date'])

    out['NPK_ratio'] = out['N'] / (out['P'] + out['K'] + 1.0)
    out['Soil_Health_Index'] = out['OC'] + out['CEC'] / 10 + out['Ca'] / 1000 + out['Mg'] / 200
    out['Micronutrient_Score'] = out[['Cu', 'Zn', 'Fe', 'Mn', 'B', 'Mo']].sum(axis=1)
    out['Weather_Stress'] = np.abs(out['Temperature'] - 25.0) + np.abs(out['Rainfall'] - 150.0) / 100.0
    out['Vegetation_Index'] = out['NDVI'] * out['LAI'] * out['Chlorophyll'] / 50.0
    out['Soil_Water_Interaction'] = out['Water_Holding_Capacity'] * out['Bulk_Density']
    out['Terrain_Risk_Proxy'] = out['Sand'] / 100 + out['Slope'] / 30 + out['Elevation'] / 2000
    out['Management_Proxy'] = out['NDVI'] + out['LAI'] / 3 + out['Chlorophyll'] / 40
    return out

df = add_engineered_features(df_raw)